[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dharshan007x/Used-car-price-intelligence-CART-/blob/main/notebooks/exploratory_analysis.ipynb)

# Used Car Price Prediction Using CART Regression Trees
### Batch 12 | Machine Learning Project
---
This notebook provides exploratory analysis, recursive binary tree splitting, depth experiments, and cost-complexity pruning.

In [ ]:
# Setup for Google Colab environment
import os, sys
if 'google.colab' in sys.modules or ('get_ipython' in globals() and 'google.colab' in str(get_ipython())):
    if not os.path.exists('Used-car-price-intelligence-CART-'):
        !git clone https://github.com/dharshan007x/Used-car-price-intelligence-CART-.git
    %cd Used-car-price-intelligence-CART-
    !pip install -q -r requirements.txt
    sys.path.insert(0, '.')
else:
    sys.path.append('..')

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
from src.data_loader import load_raw_dataset
from src.data_validation import perform_data_validation
from src.feature_engineering import engineer_features
from src.cart_model import create_cart_regressor, evaluate_cart_performance
from src.depth_analysis import run_depth_experiments
from src.pruning import run_pruning_experiments

## 1. Data Acquisition and Cleaning

In [ ]:
df_raw = load_raw_dataset('../data/raw/car_details_v3.csv')
print('Raw Dataset Dimensions:', df_raw.shape)
df_clean, audit = perform_data_validation(df_raw)
df_eng = engineer_features(df_clean)
print('Cleaned & Engineered Dimensions:', df_eng.shape)
df_eng.head()

## 2. Statistical Analysis and Feature Correlations

In [ ]:
from src.eda import compute_statistical_summary
summary = compute_statistical_summary(df_eng)
display(summary)

plt.figure(figsize=(10, 6))
sns.histplot(df_eng['selling_price'], kde=True, bins=50, color='royalblue')
plt.title('Distribution of Used Car Resale Prices (INR)')
plt.xlabel('Selling Price (₹)')
plt.show()

## 3. Tree Depth Experimentation
Comparing training vs validation performance across depths [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, None].

In [ ]:
depth_df = pd.read_csv('../reports/depth_analysis.csv')
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(depth_df['depth_label'], depth_df['train_rmse'], label='Train RMSE', linestyle='--')
plt.plot(depth_df['depth_label'], depth_df['val_rmse'], label='Validation RMSE', color='red', linewidth=2)
plt.xticks(rotation=45)
plt.title('Tree Depth vs RMSE')
plt.ylabel('RMSE (₹)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(depth_df['depth_label'], depth_df['train_r2'], label='Train R²', linestyle='--')
plt.plot(depth_df['depth_label'], depth_df['val_r2'], label='Validation R²', color='green', linewidth=2)
plt.xticks(rotation=45)
plt.title('Tree Depth vs R² Score')
plt.ylabel('R² Score')
plt.legend()

plt.tight_layout()
plt.show()

## 4. Cost-Complexity Pruning
$$R_\alpha(T) = R(T) + \alpha |T|$$

In [ ]:
pruning_df = pd.read_csv('../reports/pruning_analysis.csv')
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.semilogx(pruning_df['alpha'], pruning_df['total_nodes'], label='Total Nodes', color='purple')
plt.semilogx(pruning_df['alpha'], pruning_df['leaf_nodes'], label='Leaf Nodes', color='orange')
plt.title('Alpha vs Tree Complexity')
plt.xlabel('Alpha (Log Scale)')
plt.ylabel('Node Count')
plt.legend()

plt.subplot(1, 2, 2)
plt.semilogx(pruning_df['alpha'], pruning_df['val_rmse'], label='Validation RMSE', color='red')
plt.semilogx(pruning_df['alpha'], pruning_df['train_rmse'], label='Train RMSE', linestyle='--')
plt.title('Alpha vs Prediction Error')
plt.xlabel('Alpha (Log Scale)')
plt.ylabel('RMSE (₹)')
plt.legend()

plt.tight_layout()
plt.show()

## 5. Master Model Comparison

In [ ]:
comp_df = pd.read_csv('../reports/model_comparison.csv')
display(comp_df)